# DAR-GRPO: fresh run with complete checkpoints

This run starts GRPO from the SFT checkpoint at step zero. It uses the original 13,646-step learning-rate schedule and stops after saving a full checkpoint at step 1000. The checkpoint includes model weights, AdamW optimizer, scheduler, RNG state, and Trainer state. Temporary runtime files live in `/kaggle/tmp` to leave space for the checkpoint in `/kaggle/working`.


In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE=/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128
REPO_SOURCE=/kaggle/input/datasets/vucongaaa/dar-r1/DAR
ANNOTATION=/kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl
VIDEO_DATASET=/kaggle/input/datasets/vucongaaa/vce-original-videos

test -d "$WHEELHOUSE"
test -f "$WHEELHOUSE/ms_swift-3.12.5-py3-none-any.whl"
test -f "$WHEELHOUSE/numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
test -f "$WHEELHOUSE/scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl"
test -f "$WHEELHOUSE/trl-0.24.0-py3-none-any.whl"
test -f "$WHEELHOUSE/msgspec-0.19.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl" || {
  echo "Missing msgspec 0.19.0 CPython 3.12 wheel in the offline wheelhouse." >&2
  exit 1
}
test -f "$REPO_SOURCE/ms-swift/examples/train/grpo/dar/prepare_dar_grpo_data.py"
test -f "$REPO_SOURCE/ms-swift/examples/train/grpo/plugin/dar_plugin.py"
test -f "$ANNOTATION"
test -d "$VIDEO_DATASET"

echo "DAR inputs are attached."
python --version
python -c 'import sys; assert sys.version_info[:2] == (3, 12), sys.version'
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
set -euo pipefail

WHEELHOUSE=/kaggle/input/datasets/vucongaaa/dar-r1/offline-wheelhouse-kaggle-py312-cu128
RUNTIME=/kaggle/tmp/dar_grpo_python
mkdir -p "$RUNTIME"

# Do not install into /usr/local: Kaggle may already have NumPy/Torch loaded.
# packaging 26.3 is intentionally excluded; ms-swift 3.12.5 uses packaging 25.0.
mapfile -d '' WHEELS < <(find "$WHEELHOUSE" -maxdepth 1 -type f \
  -name '*.whl' \
  ! -name '*cp313*' \
  ! -name 'packaging-26.3*' \
  ! -name 'transformers-4.46.3*' \
  -print0 | sort -z)

test "${#WHEELS[@]}" -gt 0
python -m pip install --no-index --no-deps --upgrade --target "$RUNTIME" "${WHEELS[@]}"
test -x "$RUNTIME/bin/swift"
echo "Isolated offline runtime ready: $RUNTIME"

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/kaggle/tmp/dar_grpo_python

python - <<'PY'
import sys
import torch
import transformers
import swift
import trl
import peft
import msgspec
import datasets
import pyarrow
import modelscope
import qwen_vl_utils
import decord
import numpy
import scipy

runtime = "/kaggle/tmp/dar_grpo_python"
print("Python:", sys.version.split()[0])
print("numpy:", numpy.__version__, numpy.__file__)
print("scipy:", scipy.__version__, scipy.__file__)
print("pyarrow:", pyarrow.__version__, pyarrow.__file__)
print("torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("transformers:", transformers.__version__)
print("swift:", swift.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("modelscope:", modelscope.__version__)
print("msgspec:", msgspec.__version__, msgspec.__file__)

assert sys.version_info[:2] == (3, 12)
assert numpy.__version__ == "2.2.6"
assert scipy.__version__ == "1.16.3"
assert pyarrow.__version__ == "20.0.0"
assert datasets.__version__ == "3.6.0"
assert transformers.__version__ == "4.57.1"
assert swift.__version__ == "3.12.5"
assert trl.__version__ == "0.24.0"
assert msgspec.__version__ == "0.19.0"

assert numpy.__file__.startswith(runtime), numpy.__file__
assert scipy.__file__.startswith(runtime), scipy.__file__
assert pyarrow.__file__.startswith(runtime), pyarrow.__file__
assert msgspec.__file__.startswith(runtime), msgspec.__file__

assert torch.cuda.is_available(), "Kaggle GPU is not enabled"
PY

In [ ]:
%%bash
set -euo pipefail

SOURCE=/kaggle/input/datasets/vucongaaa/dar-r1/DAR
TARGET=/kaggle/tmp/DAR

# Always refresh the writable copy so reruns cannot retain an older helper.
mkdir -p "$TARGET"
cp -a "$SOURCE"/. "$TARGET"/

test -f "$TARGET/ms-swift/examples/train/grpo/dar/prepare_dar_grpo_data.py"
test -f "$TARGET/ms-swift/examples/train/grpo/plugin/dar_plugin.py"
grep -q -- '--video-root' "$TARGET/ms-swift/examples/train/grpo/dar/prepare_dar_grpo_data.py" || {
  echo "The attached dar-r1 dataset is stale: publish the updated GRPO data helper." >&2
  exit 1
}
echo "Writable DAR repo ready: $TARGET"

In [ ]:
from pathlib import Path
import os

# EDIT THIS PATH after attaching the SFT checkpoint as a Kaggle input.
SFT_MODEL_SOURCE = Path(
    "/kaggle/input/models/vucongaaa/sft/pytorch/default/1/checkpoint-214"
)
MODEL_ALIAS = Path("/kaggle/tmp/dar_sft_model")

if not SFT_MODEL_SOURCE.is_dir():
    raise FileNotFoundError(
        f"Edit SFT_MODEL_SOURCE: checkpoint directory not found: {SFT_MODEL_SOURCE}"
    )
if not (SFT_MODEL_SOURCE / "config.json").is_file():
    raise FileNotFoundError(f"Missing config.json under {SFT_MODEL_SOURCE}")
if not (SFT_MODEL_SOURCE / "preprocessor_config.json").is_file():
    raise FileNotFoundError(
        f"Missing preprocessor_config.json under {SFT_MODEL_SOURCE}; publish the complete SFT checkpoint"
    )
weight_files = sorted(SFT_MODEL_SOURCE.glob("*.safetensors"))
if not weight_files:
    raise FileNotFoundError(f"No *.safetensors weights under {SFT_MODEL_SOURCE}")

if os.path.lexists(MODEL_ALIAS):
    if not MODEL_ALIAS.is_symlink() or MODEL_ALIAS.resolve() != SFT_MODEL_SOURCE.resolve():
        raise RuntimeError(f"Existing alias points elsewhere: {MODEL_ALIAS}")
else:
    MODEL_ALIAS.symlink_to(SFT_MODEL_SOURCE, target_is_directory=True)

print("SFT checkpoint:", SFT_MODEL_SOURCE)
print("Weight files:", [p.name for p in weight_files])
print("Working alias:", MODEL_ALIAS, "->", MODEL_ALIAS.resolve())

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/kaggle/tmp/dar_grpo_python

python - <<'PY'
import json
from pathlib import Path
from transformers import AutoProcessor, Qwen2_5_VLConfig

model_path = "/kaggle/tmp/dar_sft_model"

raw_config = json.loads(
    (Path(model_path) / "config.json").read_text(encoding="utf-8")
)

config = Qwen2_5_VLConfig.from_pretrained(
    model_path,
    local_files_only=True,
)

processor = AutoProcessor.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=True,
    use_fast=False,
)

print("Architecture:", config.architectures)
print("Root model type:", raw_config.get("model_type"))
print("Loaded config type:", type(config).__name__, config.model_type)
print("Text config type:", config.text_config.model_type)
print("Processor:", type(processor))

assert raw_config.get("model_type") == "qwen2_5_vl"
assert raw_config.get("vision_config"), "Checkpoint khong co vision_config"
assert "Qwen2_5_VLForConditionalGeneration" in (
    raw_config.get("architectures") or []
)
assert type(config).__name__ == "Qwen2_5_VLConfig"
assert type(processor).__name__ == "Qwen2_5_VLProcessor"

print("Checkpoint Qwen2.5-VL multimodal: OK")
PY

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/kaggle/tmp/dar_grpo_python

VIDEO_DATASET=/kaggle/input/datasets/vucongaaa/vce-original-videos
ANNOTATION=/kaggle/input/datasets/vucongaaa/dar-annotation/train.jsonl
OUTPUT=/kaggle/tmp/dar_grpo/train_qwen25vl_ms_grpo.jsonl
PREPARE=/kaggle/tmp/DAR/ms-swift/examples/train/grpo/dar/prepare_dar_grpo_data.py
PROMPT=/kaggle/tmp/DAR/ms-swift/examples/train/grpo/prompt.txt

SAMPLE_VIDEO=$(find "$VIDEO_DATASET" -type f -name '06720.mp4' -print -quit)
test -n "$SAMPLE_VIDEO" || { echo "Cannot find 06720.mp4" >&2; exit 1; }
VIDEO_ROOT=$(dirname "$SAMPLE_VIDEO")
mkdir -p /kaggle/tmp/dar_grpo /kaggle/working/dar_grpo
echo "VIDEO_ROOT=$VIDEO_ROOT"

python "$PREPARE" \
  --input "$ANNOTATION" \
  --output "$OUTPUT" \
  --prompt "$PROMPT" \
  --video-root "$VIDEO_ROOT" \
  --check-videos \
  --strict

test "$(wc -l < "$ANNOTATION")" -eq 13646
test "$(wc -l < "$OUTPUT")" -eq 13646
echo "Validated all 13,646 GRPO records and video paths."

In [ ]:
from pathlib import Path
import json

data_path = Path("/kaggle/tmp/dar_grpo/train_qwen25vl_ms_grpo.jsonl")
with data_path.open(encoding="utf-8") as f:
    sample = json.loads(next(f))
assert [m["role"] for m in sample["messages"]] == ["system", "user"]
assert Path(sample["videos"][0]).is_file()
solution = json.loads(sample["solution"])
assert solution["segments"]
print("id:", sample["id"])
print("video:", sample["videos"][0])
print("roles:", [m["role"] for m in sample["messages"]])
print("ground-truth segments:", len(solution["segments"]))

In [ ]:
%%bash
set -euo pipefail
export PYTHONPATH=/kaggle/tmp/dar_grpo_python:/kaggle/tmp/DAR/ms-swift

python - <<'PY'
import importlib.util
import json
from swift.plugin import orms

plugin_path = "/kaggle/tmp/DAR/ms-swift/examples/train/grpo/plugin/dar_plugin.py"
spec = importlib.util.spec_from_file_location("dar_plugin", plugin_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

with open("/kaggle/tmp/dar_grpo/train_qwen25vl_ms_grpo.jsonl", encoding="utf-8") as f:
    row = json.loads(next(f))
completion = row["solution"]
for name in ("dar_struct", "dar_count", "dar_seg", "dar_emo", "dar_reason"):
    score = orms[name]()(completions=[completion], solution=[row["solution"]])[0]
    assert 0.0 <= score <= 1.0
    print(name, score)
print("Reward plugin smoke test passed.")
PY

In [ ]:
from pathlib import Path
callback = Path('/kaggle/tmp/dar_grpo_phase_stop.py')
callback.write_text('''from transformers import TrainerCallback
from swift.plugin import extra_callbacks

class StopAtFullCheckpoint(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step >= 1000:
            control.should_save = True
            control.should_training_stop = True
        return control

extra_callbacks.append(StopAtFullCheckpoint())
''')
print('Full-state phase stop callback:', callback)


In [ ]:
%%bash
set -euo pipefail

MODE=full  # Fresh GRPO from the SFT checkpoint; save complete training state.

export PYTHONPATH=/kaggle/tmp/dar_grpo_python:/kaggle/tmp/DAR/ms-swift
export PATH=/kaggle/tmp/dar_grpo_python/bin:$PATH
export CUDA_VISIBLE_DEVICES=0
export NPROC_PER_NODE=1
export MASTER_PORT=29501
export TOKENIZERS_PARALLELISM=false
export PYTHONUNBUFFERED=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
export FPS_MIN_FRAMES=16
export FPS_MAX_FRAMES=16
export VIDEO_MAX_PIXELS=100352

SWIFT_BIN=/kaggle/tmp/dar_grpo_python/bin/swift
MODEL_NAME=/kaggle/tmp/dar_sft_model
DATA_JSONL=/kaggle/tmp/dar_grpo/train_qwen25vl_ms_grpo.jsonl
PLUGIN_FILE=/kaggle/tmp/DAR/ms-swift/examples/train/grpo/plugin/dar_plugin.py

EXTRA_ARGS=()
case "$MODE" in
  smoke)
    OUTPUT_DIR=/kaggle/working/dar_grpo/smoke
    EXTRA_ARGS+=(--max_steps 2 --max_completion_length 512 --save_strategy no --overwrite_output_dir true)
    ;;
  full)
    OUTPUT_DIR=/kaggle/working/dar_grpo/checkpoints
    EXTRA_ARGS+=(--max_steps 13646 --max_completion_length 2400 \
      --save_strategy steps --save_steps 1000 --save_only_model false --save_total_limit 1)
    ;;
  *)
    echo "MODE must be smoke or full" >&2
    exit 1
    ;;
esac

echo "Training mode: $MODE"
echo "Output: $OUTPUT_DIR"

"$SWIFT_BIN" rlhf \
  --rlhf_type grpo \
  --model "$MODEL_NAME" \
  --model_type qwen2_5_vl \
  --template qwen2_5_vl \
  --dataset "$DATA_JSONL" \
  --train_type full \
  --freeze_vit true \
  --freeze_llm false \
  --freeze_aligner false \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --learning_rate 2e-6 \
  --optim adamw_torch \
  --warmup_ratio 0.03 \
  --lr_scheduler_type cosine \
  --weight_decay 0 \
  --bf16 true \
  --tf32 true \
  --gradient_checkpointing true \
  --attn_impl sdpa \
  --max_length 4096 \
  --truncation_strategy delete \
  --num_generations 4 \
  --temperature 0.8 \
  --top_k 50 \
  --beta 0.04 \
  --use_vllm false \
  --external_plugins "$PLUGIN_FILE" /kaggle/tmp/dar_grpo_phase_stop.py \
  --reward_funcs dar_struct dar_count dar_seg dar_emo dar_reason \
  --reward_weights 0.10 0.25 0.25 0.25 0.15 \
  --split_dataset_ratio 0 \
  --logging_steps 1 \
  --report_to none \
  --dataloader_num_workers 4 \
  --seed 1234 \
  --output_dir "$OUTPUT_DIR" \
  "${EXTRA_ARGS[@]}"

In [ ]:
%%bash
set -euo pipefail

echo "Artifacts under /kaggle/working/dar_grpo:"
du -h -d 2 /kaggle/working/dar_grpo | sort -h
find /kaggle/working/dar_grpo -maxdepth 4 -type f \
  \( -name 'trainer_state.json' -o -name 'config.json' -o -name '*.safetensors' -o -name 'completions.jsonl' \) \
  -print

COMPLETIONS=$(find /kaggle/working/dar_grpo -type f -name 'completions.jsonl' -print -quit)
if [[ -n "$COMPLETIONS" ]]; then
  echo "Last completion record:"
  tail -n 1 "$COMPLETIONS" | cut -c1-2000
fi

In [ ]:
import json
from pathlib import Path
root = Path('/kaggle/working/dar_grpo/checkpoints')
checkpoints = list(root.rglob('checkpoint-1000'))
assert len(checkpoints) == 1, checkpoints
checkpoint = checkpoints[0]
required = ['model.safetensors.index.json', 'optimizer.pt', 'scheduler.pt', 'trainer_state.json']
missing = [name for name in required if not (checkpoint / name).is_file()]
rng = list(checkpoint.glob('rng_state*.pth'))
assert not missing and rng, f'Missing training state: {missing}, RNG={rng}'
state = json.loads((checkpoint / 'trainer_state.json').read_text())
assert state['global_step'] == 1000, state['global_step']
sizes = {p.name: p.stat().st_size for p in checkpoint.iterdir() if p.is_file()}
print('FULL_CHECKPOINT_OK', checkpoint, 'step=', state['global_step'], 'files=', sizes)
